In [17]:
import numpy as np

def DKeff(d, W, Ws, Ks, b, Hs, nL):

    # Initialize arrays for diffusivity and conductivity
    D = np.zeros(nL)
    K = np.zeros(nL)
    De = np.zeros(nL)
    Ke = np.zeros(nL)

   # Define nL and dR
    nL = 10
    dR = np.array([[0.1, 0.1, 0.1], [0.1, 0.15, 0.2]])

    # Calculate dgG and dgR
    dgG = 1.0 / nL * np.ones((nL, 1))  # vegetated ground
    dgR = dR[-1, 1] / (nL / 2) * np.ones((int(nL / 2), 1))  # green roof

    dgG, dgR

    d= dgR or dgG
    # 2. Cosby et al-Chen model (1984)
    for j in range(nL):
        K[j] = Ks * (W[j] / Ws) ** (2 * b + 3)
        D[j] = b * Ks * Hs * (W[j] / Ws) ** (b + 2) / Ws

    # Compute effective values for each layer except the last one
    for j in range(nL - 1):
        Ke[j] = (d[j] + d[j+1]) / ((d[j] / K[j]) + (d[j+1] / K[j+1]))
        De[j] = (d[j] + d[j+1]) / ((d[j] / D[j]) + (d[j+1] / D[j+1]))

    # Set the last values of Ke and De
    Ke[nL-1] = K[nL-1]
    De[nL-1] = D[nL-1]

    return De, Ke

In [19]:
# Define nd, dt, and th based on previous steps
nd = 366  # number of days for simulation
dt = 300  # time step in seconds

# Create the th array
th = np.arange(0, nd + dt/3600/24, dt/3600/24) * 24

# Calculate nt based on the length of th
nt = len(th)

# Initialize WGv again to be consistent with nt and nL
WGv = Ws * np.ones((nt, nL))

WGv

array([[0.4, 0.4, 0.4, 0.4, 0.4],
       [0.4, 0.4, 0.4, 0.4, 0.4],
       [0.4, 0.4, 0.4, 0.4, 0.4],
       ...,
       [0.4, 0.4, 0.4, 0.4, 0.4],
       [0.4, 0.4, 0.4, 0.4, 0.4],
       [0.4, 0.4, 0.4, 0.4, 0.4]])

In [22]:
# Create WGv and qmG based on nt and nL
WGv = Ws * np.ones((nt, nL))  # WGv has shape (nt, nL)
qmG = np.zeros((1, nL))  # Ensure qmG has the same shape as the first row of WGv

# Set qmG(1, :) = 24
qmG[0, :] = 24

# Perform the operation: WGv(1, :) = qmG(1, :) / 100
WGv[0, :] = qmG[0, :] / 100

WGv, qmG


(array([[0.24, 0.24, 0.24, 0.24, 0.24],
        [0.4 , 0.4 , 0.4 , 0.4 , 0.4 ],
        [0.4 , 0.4 , 0.4 , 0.4 , 0.4 ],
        ...,
        [0.4 , 0.4 , 0.4 , 0.4 , 0.4 ],
        [0.4 , 0.4 , 0.4 , 0.4 , 0.4 ],
        [0.4 , 0.4 , 0.4 , 0.4 , 0.4 ]]),
 array([[24., 24., 24., 24., 24.]]))

In [27]:
# Re-run the DKeff function with updated inputs
def DKeff(d, W, Ws, Ks, b, Hs, nL):
    """
    Purpose:
        Compute effective hydraulic diffusivity and conductivity.

    Parameters:
        d : numpy array
            Thickness of each layer.
        W : numpy array
            Water content of each layer.
        Ws : float
            Saturated water content.
        Ks : float
            Saturated soil conductivity.
        b : float
            Fitting parameter for the soil.
        Hs : float
            Soil water retention parameter.
        nL : int
            Number of layers.

    Returns:
        De : numpy array
            Effective hydraulic diffusivity for each layer.
        Ke : numpy array
            Effective hydraulic conductivity for each layer.
    """

    # Initialize arrays for diffusivity and conductivity
    D = np.zeros(nL)
    K = np.zeros(nL)
    De = np.zeros(nL)
    Ke = np.zeros(nL)

    # Compute K and D for each layer
    for j in range(nL):
        K[j] = Ks * (W[j] / Ws) ** (2 * b + 3)
        D[j] = b * Ks * Hs * (W[j] / Ws) ** (b + 2) / Ws

    # Compute effective values for each layer except the last one
    for j in range(nL - 1):
        Ke[j] = (d[j] + d[j+1]) / ((d[j] / K[j]) + (d[j+1] / K[j+1]))
        De[j] = (d[j] + d[j+1]) / ((d[j] / D[j]) + (d[j+1] / D[j+1]))

    # Set the last values of Ke and De
    Ke[nL-1] = K[nL-1]
    De[nL-1] = D[nL-1]

    return De, Ke

# Example usage:
nL = 5  # Adjusted for dgR
dR = np.array([[0.1, 0.1, 0.1], [0.1, 0.15, 0.2]])

# Calculate dgR
dgR = dR[-1, 1] / (nL / 2) * np.ones((int(nL), 1))  # green roof

# Example input data for W, Ws, Ks, b, Hs
Ws = 0.4  # saturated water content
Ks = 10   # saturated soil conductivity
b = 4     # fitting parameter for the soil
Hs = 2    # soil water retention parameter

# Use WGv for water content and make sure it's consistent with nL
W = WGv[:, :nL]  # Adjust WGv to match the shape

# Call the DKeff function with dgR
De, Ke = DKeff(dgR.flatten(), W[0], Ws, Ks, b, Hs, nL)

De, Ke


(array([9.3312, 9.3312, 9.3312, 9.3312, 9.3312]),
 array([0.03627971, 0.03627971, 0.03627971, 0.03627971, 0.03627971]))